# 00C Demo Setup — reset and prepare Day 1

Run after installing or upgrading the package and configuring 00B. This is the single reset entrypoint for 02A–02C. It previews exact physical assets and all 14 metadata tables, requires confirmation, restores Day 1, and verifies the reset. Keep schedules and interactive demo executions stopped throughout.

**Destructive: dedicated demo environment only. Never run against unrelated production assets.** Shared Governance and immutable deployment history are preserved; ambiguous ownership blocks reset. Run the complete notebook from the top, never just the source preparation cells.


## 1. Load the shared Fabric configuration

The setup notebook uses the same workspace-local `00_env_config` as the rest of the Guided Demo.


In [1]:
from fabricops_kit import setup_notebook

### Use this when the 00_env_config notebook is in the same workspace as this notebook
context = setup_notebook()

### Advanced you can run the 00_env_config notebook that is in another workspace, 
### make sure you have workspace execution rights to that workspace for this to work
### The caller workspace is detected automatically. Configuration workspace example: "<configuration-workspace-id>"
# Advanced: load 00_env_config from another workspace:
# context = setup_notebook(
#     config_workspace_id="<configuration-workspace-id>",
# )


## 2. Import the FabricOps I/O helpers


In [ ]:
from fabricops_kit import (
    read_lakehouse_csv,
    read_lakehouse_excel,
    read_lakehouse_json,
    read_lakehouse_parquet,
    read_lakehouse_table,
    read_warehouse_query,
    read_warehouse_table,
    write_lakehouse_table,
    write_warehouse_table,
)


## Preview and confirm the demo reset

Import 02A, 02B and 02C with their repository notebook names before starting. Their Fabric Notebook IDs are the exact Ungoverned Pipeline IDs; both 02B executions share one Notebook ID.

Stop schedules and interactive executions before proceeding, then set `DEMO_EXECUTIONS_STOPPED=True`. The live job check and incomplete Run check add fail-closed protections; they cannot lock out a new interactive execution. Keep the demo stopped until `RESET COMPLETE`.

For verified retired demo notebook copies only, add exact Fabric IDs below. Deleted copies remain historical owners; only an explicit Fabric ItemNotFound response for the item permits skipping its live-job query. An incomplete Run is blocked until its owner confirms it has stopped; add that exact Run ID only after checking Fabric Monitor. Never use these settings to bypass an active job or unrelated ownership.

Shared Contracts, activations and immutable promotions are not erased. If they reference these demo identities, use a dedicated Ungoverned environment. Run the installed package metadata setup/migration in 00B first. 00C checks required column names and types; it never migrates schemas. Keep schema migrations stopped until verification finishes. Preview and counts include all current tables; zero removals are intentional.


In [ ]:
DEMO_EXECUTIONS_STOPPED = False  # Set True only after stopping schedules and interactive demo executions.
EXTRA_DEMO_NOTEBOOK_IDS = []  # Exact IDs of verified retired demo copies, if any.
STOPPED_DEMO_RUN_IDS = []  # Exact incomplete Run IDs confirmed terminated in Fabric Monitor.


In [ ]:
# Demo-only cleanup. These are existing foundations, not new notebook-facing APIs.
import json
from collections import Counter
from uuid import UUID, uuid4
from urllib.error import HTTPError

from delta.tables import DeltaTable
from pyspark.sql import functions as F

from fabricops_kit import resolve_table_id
from fabricops_kit.activated_pipeline.shared import FabricNotebookClient
from fabricops_kit.config.metadata_schemas import metadata_table_physical_schema, metadata_table_schema_registry
from fabricops_kit.config.shared import resolve_fabric_context
from fabricops_kit.io.shared import (
    execute_warehouse_sql,
    resolve_configured_lakehouse_table,
    resolve_warehouse_name,
    retry_delta_metadata,
)

_DEMO_SOURCES = {
    ('Bronze', 'orders'): ('Demo/orders.csv', 120),
    ('Bronze', 'products'): ('Demo/products.csv', 8),
    ('Gold', 'order_history'): ('Demo/order_history.csv', 180),
    ('Bronze', 'inventory_movements'): ('Demo/inventory_day1.csv', 8),
    ('Bronze', 'product_master_updates'): ('Demo/products_day1.csv', 8),
}
_DEMO_OUTPUTS = (
    ('Silver', 'curated_orders'), ('Gold', 'customer_summary'),
    ('Silver', 'inventory_movements_append_all'), ('Silver', 'inventory_movements_append_new_only'),
    ('Silver', 'product_master_scd1'), ('Silver', 'product_master_scd2'),
)
_DEMO_NAMES = {'02A_full_refresh_demo', '02B_incremental_append_demo', '02C_scd_demo'}
# Children before parents; Run history is retained until the other deletions succeed.
_DEMO_DELETE_ORDER = (
    'METADATA_GUARDRAIL_RESULTS', 'METADATA_PROCESSING_STATE',
    'METADATA_DATA_PROFILED_FREQUENCY', 'METADATA_DATA_PROFILED',
    'METADATA_DATA_LINEAGE', 'METADATA_DATA_ACCESS', 'METADATA_DATA_CATALOGUE', 'METADATA_PIPELINE_RUN',
)


def _demo_row_counts(rows):
    """Compare complete records, including duplicates and immutable audit values."""
    return Counter(json.dumps(row, sort_keys=True, default=str) for row in rows)


def _demo_operations(row):
    """Read the physical JSON STRING Run contract; reject malformed history."""
    value = row.get('operations')
    operations = json.loads(value) if isinstance(value, str) else value
    if not isinstance(operations, list) or any(not isinstance(op, dict) for op in operations):
        raise RuntimeError(f"Invalid operations for run {row.get('run_id')}; reconcile history before reset.")
    return operations


def _demo_payload_tables(value):
    """Inspect structured immutable packages for exact table references, never substrings."""
    if isinstance(value, dict):
        result = {value['table_id']} if isinstance(value.get('table_id'), str) else set()
        for child in value.values():
            if isinstance(child, (dict, list)):
                result.update(_demo_payload_tables(child))
        return result
    if isinstance(value, list):
        return set().union(*(_demo_payload_tables(child) for child in value)) if value else set()
    return set()


def _demo_scope(rows, workspace, table_ids, notebook_ids, physical_workspaces, stopped_runs=(), bindings=None):
    """Fail closed on shared ownership and return exact disposable demo rows."""
    selected = {table: [] for table in rows}
    runs = rows['METADATA_PIPELINE_RUN']
    run_ids = set()
    for row in runs:
        if row.get('workspace_id') != workspace:
            continue
        operations = _demo_operations(row)
        profiling = row.get('profiling') or []
        profiling = json.loads(profiling) if isinstance(profiling, str) else profiling
        if not isinstance(profiling, list) or any(not isinstance(event, dict) for event in profiling):
            raise RuntimeError(f"Invalid profiling for run {row.get('run_id')}; reconcile history before reset.")
        touched = {op.get('table_id') for op in operations}
        touched |= {event.get('table_id') for event in profiling} - {None, ''}
        referenced = touched | {tid for op in operations for tid in op.get('source_table_ids', [])}
        referenced |= {op.get('incremental_target_table_id') for op in operations} - {None}
        if row.get('pipeline_id') not in notebook_ids and not referenced.intersection(table_ids):
            continue
        if row.get('pipeline_id') not in notebook_ids or not referenced.issubset(table_ids):
            raise RuntimeError(f"Run {row.get('run_id')} has ambiguous demo ownership. Check its exact Notebook ID/operations; "
                               "add only a verified retired demo ID to EXTRA_DEMO_NOTEBOOK_IDS, or use a dedicated environment.")
        if bindings is not None:
            for operation in operations:
                if operation.get('destination') != bindings[operation['table_id']]:
                    raise RuntimeError(f"Run {row.get('run_id')} uses different physical store mappings; preserve its history.")
        if row.get('execution_mode') != 'ungoverned' or row.get('promotion_id') or row.get('approval_id'):
            raise RuntimeError('Governed or historical non-Ungoverned demo Run found; preserve it and use a dedicated environment.')
        if not row.get('completed_at') and row.get('run_id') not in stopped_runs:
            raise RuntimeError(f"Run {row.get('run_id')} may still be active. Stop it in Fabric; only after confirming termination "
                               "add its exact ID to STOPPED_DEMO_RUN_IDS and retry.")
        selected['METADATA_PIPELINE_RUN'].append(row)
        run_ids.add(row['run_id'])

    for table, records in rows.items():
        for row in records:
            if table == 'METADATA_DATA_CONTRACT':
                if row.get('evidence_workspace_id') == workspace and row.get('table_id') in table_ids:
                    raise RuntimeError('Demo Data Contract history exists. Preserve Governance records; use a dedicated Ungoverned environment.')
            elif table == 'METADATA_DATA_CONTRACT_ACTIVATION':
                if row.get('table_id') in table_ids and row.get('workspace_id') in physical_workspaces:
                    raise RuntimeError('Demo Contract Activation history exists; use a dedicated Ungoverned environment.')
            elif table in {'METADATA_PIPELINE_REGISTRY', 'METADATA_PIPELINE_ACTIVATION'}:
                bound = {row.get('pipeline_id'), row.get('source_notebook_id'), row.get('governed_notebook_id')}
                payload = row.get('operations_json') if table == 'METADATA_PIPELINE_REGISTRY' else row.get('package_json')
                same_workspace = row.get('execution_workspace_id') == workspace
                shared_tables = same_workspace and bool(_demo_payload_tables(json.loads(payload) if isinstance(payload, str) else payload).intersection(table_ids))
                if shared_tables or (bound.intersection(notebook_ids) and (same_workspace
                                                        or row.get('source_workspace_id') == workspace)):
                    raise RuntimeError('Demo promotion/approval history exists; immutable Governance and Registry records are preserved.')
            elif table in _DEMO_DELETE_ORDER and table != 'METADATA_PIPELINE_RUN':
                if row.get('workspace_id') != workspace:
                    continue
                endpoints = {row.get('table_id'), row.get('source_table_id'), row.get('target_table_id')} - {None, ''}
                related = (bool(endpoints.intersection(table_ids)) or row.get('run_id') in run_ids
                           or row.get('pipeline_id') in notebook_ids or row.get('_notebook_id') in notebook_ids)
                if not related:
                    continue
                if not endpoints.issubset(table_ids):
                    raise RuntimeError(f'{table} links demo and unrelated tables; reconcile ownership before reset.')
                if table == 'METADATA_DATA_CATALOGUE' and row.get('writer_notebook_id'):
                    if row['writer_notebook_id'] not in notebook_ids or row.get('writer_workspace_id') != workspace:
                        raise RuntimeError('Demo Catalogue records have another writer owner; use a dedicated environment.')
                if row.get('pipeline_id') and row['pipeline_id'] not in notebook_ids:
                    raise RuntimeError(f'{table} belongs to another pipeline; use a dedicated environment.')
                if row.get('run_id') and row['run_id'] not in run_ids:
                    # Interrupted cleanup can leave no Run parent, but exact demo notebook ownership must remain.
                    if row.get('_notebook_id') not in notebook_ids:
                        raise RuntimeError(f'{table} has an unknown run owner; reconcile before reset.')
                if row.get('_notebook_id') not in notebook_ids:
                    raise RuntimeError(f'{table} has an unknown notebook owner; reconcile before reset.')
                if table == 'METADATA_PROCESSING_STATE' and row.get('record_type') not in {
                    'source_observation', 'incremental_checkpoint', 'drift_baseline', 'full_source_baseline', 'publication_receipt'
                }:
                    raise RuntimeError('Unknown demo Processing State type; review ownership and dependencies before reset.')
                selected[table].append(row)

    # Detect references from retained records before deleting any parent record.
    links = {
        'METADATA_DATA_PROFILED_FREQUENCY': ('METADATA_DATA_PROFILED', ('profile_id', 'profile_snapshot_id')),
        'METADATA_PROCESSING_STATE': ('METADATA_PROCESSING_STATE', ('publication_id', 'observation_id')),
        'METADATA_GUARDRAIL_RESULTS': ('METADATA_PIPELINE_RUN', ('run_id',)),
    }
    for child, (parent, fields) in links.items():
        retained = _demo_row_counts(rows[child]) - _demo_row_counts(selected[child])
        for field in fields:
            removed_ids = {r.get(field) for r in selected[parent]} - {None, ''}
            for row in rows[child]:
                if (row.get('workspace_id') == workspace and row.get(field) in removed_ids
                        and json.dumps(row, sort_keys=True, default=str) in retained):
                    raise RuntimeError(f'{child}.{field} references demo metadata from a retained record; reset blocked.')
    return selected


def _demo_metadata_rows(schemas=None):
    """Validate every physical schema before allowing any demo deletion."""
    result = {}
    registry = metadata_table_schema_registry()
    for table, expected in registry.items():
        schema = metadata_table_physical_schema(_demo_config, table)
        try:
            frame = read_lakehouse_table(table, store='Metadata', schema=schema)
        except Exception as exc:
            raise RuntimeError(f'Cannot read {schema}.{table}. Check the Metadata mapping, permissions and 00B metadata setup; reset blocked.') from exc
        actual = {field.name: field.dataType.simpleString() for field in frame.schema}
        incompatible = [field.name for field in expected
                        if actual.get(field.name) != field.dataType.simpleString()]
        if incompatible:
            raise RuntimeError(f'{table} has missing columns or incompatible types: {", ".join(incompatible)}. '
                               'Run the installed package metadata setup/migration in 00B, then restart 00C. No reset is permitted.')
        if schemas is not None:
            schemas[table] = frame.schema
        result[table] = [row.asDict(recursive=True) for row in frame.collect()]
    return result


def _demo_delete_rows(table, records):
    """Delete complete reviewed rows and measure only this reset's tagged Delta commit."""
    if not records:
        return 0
    schema = metadata_table_physical_schema(_demo_config, table)
    if read_lakehouse_table(table, store='Metadata', schema=schema).schema != _demo_schemas[table]:
        raise RuntimeError(f'{table} schema changed after preview. Restart 00C after metadata migration finishes.')
    _, _, _, path = resolve_configured_lakehouse_table('Metadata', table, schema)
    session = spark.newSession()
    for setting in ('spark.sql.session.timeZone', 'spark.sql.caseSensitive',
                    'spark.sql.ansi.enabled', 'spark.sql.storeAssignmentPolicy', 'spark.sql.shuffle.partitions'):
        session.conf.set(setting, spark.conf.get(setting))
    marker = f'fabricops-demo-reset:{uuid4()}'
    session.conf.set('spark.databricks.delta.commitInfo.userMetadata', marker)
    # Use the reviewed physical schema, including supported extension columns.
    frame = session.createDataFrame(records, schema=_demo_schemas[table])
    quoted = ['`' + col.replace('`', '``') + '`' for col in frame.columns]
    condition = ' AND '.join(f'target.{col} <=> source.{col}' for col in quoted)
    # Repeated cleanup can have no remaining matches; Delta may skip a no-op commit entirely.
    target = DeltaTable.forPath(session, path).toDF().alias('target')
    if not target.join(frame.alias('source'), F.expr(condition), 'left_semi').limit(1).count():
        return 0
    retry_delta_metadata(lambda: DeltaTable.forPath(session, path).alias('target')
                         .merge(frame.dropDuplicates().alias('source'), condition).whenMatchedDelete().execute())
    commits = (DeltaTable.forPath(session, path).history()
               .where((F.col('userMetadata') == marker) & (F.col('operation') == 'MERGE'))
               .select('operationMetrics').collect())
    if len(commits) != 1:
        raise RuntimeError(f'Cannot verify the exact {table} deletion commit. Restart 00C and inspect Delta history.')
    metrics = commits[0]['operationMetrics']
    if int(metrics['numTargetRowsInserted']) or int(metrics['numTargetRowsUpdated']):
        raise RuntimeError(f'{table} reset unexpectedly inserted or updated rows; inspect its tagged Delta commit.')
    return int(metrics['numTargetRowsDeleted'])


def _demo_assert_source(actual, expected):
    """Compare full row multisets after normalizing Warehouse connector data types."""
    actual = actual.select(*[F.col(field.name).cast(field.dataType).alias(field.name) for field in expected.schema])
    if actual.exceptAll(expected).limit(1).count() or expected.exceptAll(actual).limit(1).count():
        raise RuntimeError('Day 1 source verification failed: persisted content differs from the fixture. Rerun 00C.')


def _demo_check_jobs():
    """Check live jobs; only explicitly verified retired IDs may be confirmed missing."""
    for notebook_id in _demo_notebook_ids:
        if notebook_id in _demo_extra_notebook_ids and notebook_id not in _demo_listed_notebook_ids:
            # A missing jobs endpoint alone does not establish that the notebook was deleted.
            try:
                _demo_client.request('GET', f'/workspaces/{context.workspace_id}/items/{notebook_id}')
            except HTTPError as exc:
                try:
                    error = json.loads(exc.read()) if exc.code == 404 else {}
                except (ValueError, UnicodeDecodeError):
                    error = {}
                if exc.code == 404 and isinstance(error, dict) and error.get('errorCode') == 'ItemNotFound':
                    # Metadata scope still requires every incomplete historical Run to be confirmed stopped.
                    continue
                raise RuntimeError(f'Cannot establish whether retired demo notebook {notebook_id} exists. '
                                   'Check workspace/item permissions and Fabric availability before resetting.') from exc
        path = f'/workspaces/{context.workspace_id}/items/{notebook_id}/jobs/instances'
        while path:
            payload = _demo_client.request('GET', path)
            for job in payload['value']:
                if job['status'] not in {'Completed', 'Failed', 'Cancelled', 'Deduped'}:
                    raise RuntimeError(f'Demo notebook {notebook_id} has a non-terminal Fabric job. Stop it before resetting.')
            path = payload.get('continuationUri')


def _demo_remove_outputs():
    """Remove exact configured output objects and their physical publication markers."""
    import notebookutils
    for store, table in _DEMO_OUTPUTS:
        if store == 'Gold':
            gold = context.paths[store]
            execute_warehouse_sql(spark, gold,
                "IF OBJECT_ID(N'demo.customer_summary', N'U') IS NOT NULL DROP TABLE [demo].[customer_summary]; "
                'SELECT 1 AS demo_reset_dropped;', database_name=resolve_warehouse_name(gold, store))
        else:
            _, _, _, path = resolve_configured_lakehouse_table(store, table, 'demo')
            qualified = f'{_demo_silver_catalog}.`{table}`'
            if spark.catalog.tableExists(qualified):
                location_rows = spark.sql(f'DESCRIBE TABLE EXTENDED {qualified}').where("col_name = 'Location'").collect()
                if len(location_rows) != 1 or location_rows[0]['data_type'].rstrip('/') != path.rstrip('/'):
                    raise RuntimeError(f'Catalog location mismatch for {store}.demo.{table}; reset blocked.')
                spark.sql(f'DROP TABLE {qualified}')
            if notebookutils.fs.exists(path) and not notebookutils.fs.rm(path, True):
                raise RuntimeError(f'Could not remove {store}.demo.{table}; check permissions and retry 00C.')
        print(f'Removed output (missing is OK): {store}.demo.{table}')


def _demo_verify_outputs():
    """Fail if any of the six physical outputs still exists."""
    import notebookutils
    for store, table in _DEMO_OUTPUTS:
        if store == 'Gold':
            exists = read_warehouse_query(
                "SELECT COUNT(*) AS n FROM sys.tables t JOIN sys.schemas s ON t.schema_id=s.schema_id "
                "WHERE s.name='demo' AND t.name='customer_summary'", store='Gold').first()['n'] != 0
        else:
            _, _, _, path = resolve_configured_lakehouse_table(store, table, 'demo')
            exists = notebookutils.fs.exists(path) or spark.catalog.tableExists(f'{_demo_silver_catalog}.`{table}`')
        if exists:
            raise RuntimeError(f'Reset incomplete: {store}.demo.{table} remains. Check permissions and rerun 00C.')


### Preview (reset-preview)

Run this cell and review the exact configured assets and metadata counts before confirming.


In [ ]:
# Preview only. No data or metadata is changed by this cell.
_demo_confirmed = False
if context.readiness_status != 'ready':
    raise RuntimeError('Notebook setup is not ready. Correct 00_env_config and rerun 00C.')
_demo_config, _demo_workspace, _ = resolve_fabric_context()
for _key, _kind in [('Bronze', 'lakehouse'), ('Silver', 'lakehouse'), ('Gold', 'warehouse'), ('Metadata', 'lakehouse')]:
    _store = context.paths[_key]
    if _store.kind != _kind or (_kind == 'lakehouse' and not _store.schema_enabled):
        raise RuntimeError(f'{_key} must be a {_kind} with the demo schema configuration (Lakehouses require schemas).')
if len({(context.paths[k].workspace_id, context.paths[k].item_id) for k in ('Bronze', 'Silver', 'Gold', 'Metadata')}) != 4:
    raise RuntimeError('Demo business stores and Metadata must resolve to distinct physical items.')

_demo_client = FabricNotebookClient()
_demo_notebooks = []
_demo_path = f'/workspaces/{context.workspace_id}/notebooks'
while _demo_path:
    _payload = _demo_client.request('GET', _demo_path)
    _demo_notebooks.extend(_payload['value'])
    _demo_path = _payload.get('continuationUri')
_demo_notebook_ids = set()
for _name in _DEMO_NAMES:
    _matches = [item['id'] for item in _demo_notebooks if item['displayName'] == _name]
    if len(_matches) != 1:
        raise RuntimeError(f'Import exactly one {_name} notebook with its original name before running 00C.')
    _demo_notebook_ids.add(_matches[0])
_demo_listed_notebook_ids = {item['id'] for item in _demo_notebooks}
_demo_extra_notebook_ids = {str(UUID(value)) for value in EXTRA_DEMO_NOTEBOOK_IDS}
_demo_notebook_ids.update(_demo_extra_notebook_ids)
_demo_table_ids = {resolve_table_id(store=store, schema='demo', table_name=table)
                   for store, table in [*_DEMO_SOURCES, *_DEMO_OUTPUTS]}
_demo_bindings = {resolve_table_id(store=store, schema='demo', table_name=table):
                  dict(workspace_id=context.paths[store].workspace_id, item_id=context.paths[store].item_id,
                       kind=context.paths[store].kind, schema='demo', table_name=table)
                  for store, table in [*_DEMO_SOURCES, *_DEMO_OUTPUTS]}
_demo_physical_workspaces = {context.paths[store].workspace_id for store, _ in [*_DEMO_SOURCES, *_DEMO_OUTPUTS]}

# Resolve the Spark catalog name, then validate its storage location before DROP.
_silver = context.paths['Silver']
_silver_workspace_name = _demo_client.request('GET', f'/workspaces/{_silver.workspace_id}')['displayName']
_silver_name = _demo_client.request('GET', f'/workspaces/{_silver.workspace_id}/items/{_silver.item_id}')['displayName']
_demo_silver_catalog = '.'.join('`' + value.replace('`', '``') + '`' for value in (_silver_workspace_name, _silver_name, 'demo'))
_demo_check_jobs()
_demo_schemas = {}
_demo_before = _demo_metadata_rows(schemas=_demo_schemas)
_demo_preserved_tables = {'METADATA_DATA_STEWARD', 'METADATA_DATA_AGREEMENT', 'METADATA_DATA_CONTRACT',
                          'METADATA_DATA_CONTRACT_ACTIVATION', 'METADATA_PIPELINE_ACTIVATION', 'METADATA_PIPELINE_REGISTRY'}
if set(_demo_before) != set(_DEMO_DELETE_ORDER) | _demo_preserved_tables:
    raise RuntimeError('Metadata inventory changed. Review all table ownership before using this demo reset with this package.')
_demo_selected = _demo_scope(_demo_before, context.workspace_id, _demo_table_ids, _demo_notebook_ids,
                             _demo_physical_workspaces, STOPPED_DEMO_RUN_IDS, bindings=_demo_bindings)
# Delta equality cannot match map-valued extensions. Block only tables with selected demo rows.
for _table, _records in _demo_selected.items():
    if _records and any('map<' in field.dataType.simpleString() for field in _demo_schemas[_table]):
        raise RuntimeError(f'{_table} has a map-valued extension that cannot be matched safely; reset blocked for review.')
# Read and materialize all five fixtures before any destructive operation.
_demo_day1 = {}
for (_store, _table), (_file, _count) in _DEMO_SOURCES.items():
    _frame = read_lakehouse_csv(_file, store='Bronze', header=True, inferSchema=True).cache()
    if _frame.count() != _count:
        raise RuntimeError(f'{_file} must contain {_count} rows. Restore the repository fixture and retry.')
    _demo_day1[(_store, _table)] = _frame
print('WARNING: dedicated demo environment only. Stop schedules and interactive executions until verification finishes.')
for _key in ('Bronze', 'Silver', 'Gold', 'Metadata'):
    _store = context.paths[_key]
    print(f'{_key}: workspace={_store.workspace_id}, item={_store.item_id}')
print(f'Exact demo Notebook/Pipeline IDs: {sorted(_demo_notebook_ids)}')
for _store, _table in _DEMO_SOURCES:
    print(f'RESTORE Day 1: {_store}.demo.{_table}')
for _store, _table in _DEMO_OUTPUTS:
    print(f'DROP output: {_store}.demo.{_table}')
for _table in _demo_before:
    print(f'{_table}: remove {len(_demo_selected[_table])}; preserve {len(_demo_before[_table]) - len(_demo_selected[_table])}')


### Confirm and apply (reset-confirmation / reset-apply)

After reviewing the preview, enter `RESET DEMO` in the next cell and execute it. Then run the application cell. Keep demo executions stopped.


In [ ]:
# After reviewing reset-preview, enter RESET DEMO below and execute this cell, then reset-apply.
# A fresh application consumes this confirmation. No stdin/widget support is required.
RESET_CONFIRMATION = ""


In [ ]:
_demo_confirmed = False
if RESET_CONFIRMATION != 'RESET DEMO':
    raise RuntimeError('Reset cancelled. Review reset-preview, enter RESET DEMO in reset-confirmation and execute it, then apply.')
RESET_CONFIRMATION = ''  # Consume the reviewed confirmation before any mutation.
if not DEMO_EXECUTIONS_STOPPED:
    raise RuntimeError('Stop all demo schedules and interactive executions, set DEMO_EXECUTIONS_STOPPED=True, then rerun the preview.')
_demo_check_jobs()
_demo_current_schemas = {}
_demo_now = _demo_metadata_rows(schemas=_demo_current_schemas)
if _demo_current_schemas != _demo_schemas:
    raise RuntimeError('Metadata schemas changed since preview. Finish metadata migration and restart 00C.')
_demo_current_selected = _demo_scope(_demo_now, context.workspace_id, _demo_table_ids, _demo_notebook_ids,
                                     _demo_physical_workspaces, STOPPED_DEMO_RUN_IDS, bindings=_demo_bindings)
if any(_demo_row_counts(_demo_current_selected[t]) != _demo_row_counts(_demo_selected[t]) for t in _demo_selected):
    raise RuntimeError('Demo metadata changed since preview. Keep demo executions stopped and rerun 00C from the top.')
_demo_removed = {table: 0 for table in _demo_before}
_demo_confirmed = True
# Remove metadata first, keeping Run history until all children are removed.
# Each statement is exact and idempotent; failures propagate, without RESET COMPLETE.
try:
    for _table in _DEMO_DELETE_ORDER:
        _demo_action = _table
        _demo_removed[_table] = _demo_delete_rows(_table, _demo_selected[_table])
        if _demo_removed[_table] != len(_demo_selected[_table]):
            raise RuntimeError('Reviewed demo rows changed during cleanup; stop demo writers and restart 00C.')
    _demo_action = 'demo output tables'
    _demo_remove_outputs()
except Exception as exc:
    _demo_confirmed = False
    raise RuntimeError(f'Demo reset incomplete while cleaning {_demo_action}. Resolve the cause and rerun 00C from the top.') from exc


## 3. Read the same "Orders" dataset via CSV, JSON, Parquet, and Excel.

All four files contain the same 120 logical Orders rows. Paths are relative to the configured Lakehouse `Files` area, so `Demo/orders.csv` resolves under `Files/Demo/orders.csv`.


In [ ]:
orders_csv_df = read_lakehouse_csv(
    "Demo/orders.csv",
    store="Bronze",
    
    header=True,
    inferSchema=True,
)

#display(orders_csv_df)


In [ ]:
orders_json_df = read_lakehouse_json(
    "Demo/orders.json",
    store="Bronze",
    
)
#display(orders_json_df)


In [ ]:
orders_parquet_df = read_lakehouse_parquet(
    "Demo/orders.parquet",
    store="Bronze",
    
)

#display(orders_parquet_df)


In [ ]:
orders_excel_df = read_lakehouse_excel(
    "Demo/orders.xlsx",
    store="Bronze",
    
)
#display(orders_excel_df)


## 4. Read and write the demo inputs into the lakehouse and warehouse tables

`products.csv` becomes the Lakehouse lookup table. `order_history.csv` becomes the Warehouse history table.


In [ ]:
if not _demo_confirmed:
    raise RuntimeError('Run the reset preview and confirmation before restoring sources.')

# Write the Orders table into lakehouse 

write_lakehouse_table(
    _demo_day1[('Bronze', 'orders')],
    table_name="orders",
    store="Bronze",
    schema="demo",
    mode="overwrite",
)

#display(orders_csv_df)

# Optional Read the table that is written in the lakehouse to see its written 
orders_table_df = read_lakehouse_table(
    table_name="orders",
    store="Bronze",
    schema="demo",
    
)

#display(orders_table_df)


In [ ]:
if not _demo_confirmed:
    raise RuntimeError('Run the reset preview and confirmation before restoring sources.')

# Read and Write the Product table into lakehouse 

products_df = read_lakehouse_csv(
    "Demo/products.csv",
    store="Bronze",
    
    header=True,
    inferSchema=True,
)

#display(products_df)

write_lakehouse_table(
    _demo_day1[('Bronze', 'products')],
    table_name="products",
    store="Bronze",
    schema="demo",
    mode="overwrite",
)

products_table_df = read_lakehouse_table(
    table_name="products",
    store="Bronze",
    schema="demo",
    
)

#display(products_table_df)


In [ ]:
if not _demo_confirmed:
    raise RuntimeError('Run the reset preview and confirmation before restoring sources.')

order_history_df = read_lakehouse_csv(
    "Demo/order_history.csv",
    store="Bronze",
    
    header=True,
    inferSchema=True,
)

#display(order_history_df)

write_warehouse_table(
    _demo_day1[('Gold', 'order_history')],
    schema="demo",
    table_name="order_history",
    store="Gold",
    mode="overwrite",
)

# read_warehouse_table() reads the full table. 
order_history_table_df = read_warehouse_table(
    schema="demo",
    table_name="order_history",
    store="Gold",
    
)

#display(order_history_table_df)

# read_warehouse_query() pushes SQL to the Warehouse before returning the result to Spark. 
order_history_sample_df = read_warehouse_query(
    "SELECT TOP 5 * FROM demo.order_history ORDER BY 1",
    store="Gold",
    
)

#display(order_history_sample_df)


## 5. Restore Day 1 for append and SCD demos

Overwrite the two dedicated Bronze sources after the confirmed reset. Run 00C only when starting or restarting the complete Day 1–3 sequence. Do not run it between days.


In [ ]:
if not _demo_confirmed:
    raise RuntimeError('Run the reset preview and confirmation before restoring sources.')

for path, table in [
    ("Demo/inventory_day1.csv", "inventory_movements"),
    ("Demo/products_day1.csv", "product_master_updates"),
]:
    day1_df = _demo_day1[('Bronze', table)]
    write_lakehouse_table(
        day1_df, table_name=table, store="Bronze", schema="demo", mode="overwrite",
    )
    print(f"Prepared Bronze demo.{table}: {day1_df.count()} Day 1 rows")


## 6. Verify the complete reset

Compare all five persisted sources to Day 1, check six outputs are absent, and verify exact metadata removals and preservation. An exception means the reset is incomplete: resolve its cause and rerun 00C from the top.


In [ ]:
if not _demo_confirmed:
    raise RuntimeError('Reset was not confirmed. Run 00C from the top.')
for (_store, _table), _expected in _demo_day1.items():
    _reader = read_warehouse_table if _store == 'Gold' else read_lakehouse_table
    _demo_assert_source(_reader(table_name=_table, store=_store, schema='demo'), _expected)
_demo_verify_outputs()
_demo_check_jobs()
_demo_final_schemas = {}
_demo_after = _demo_metadata_rows(schemas=_demo_final_schemas)
if _demo_final_schemas != _demo_schemas:
    raise RuntimeError('Metadata schemas changed during reset. Finish metadata migration and restart 00C.')
_demo_residual = _demo_scope(_demo_after, context.workspace_id, _demo_table_ids, _demo_notebook_ids,
                             _demo_physical_workspaces, STOPPED_DEMO_RUN_IDS, bindings=_demo_bindings)
for _table in _demo_before:
    if _demo_residual[_table]:
        raise RuntimeError(f'Reset incomplete: {_table} still has demo-owned rows. Stop executions and rerun 00C.')
    if _demo_removed[_table] != len(_demo_selected[_table]):
        raise RuntimeError(f'{_table} deletion count differs from the reviewed demo rows. Inspect the reset Delta commit.')
    print(f'{_table}: removed {_demo_removed[_table]}; remaining {len(_demo_after[_table])}')
for _frame in _demo_day1.values():
    _frame.unpersist()
_demo_confirmed = False  # Another application requires another preview and explicit confirmation.
print('RESET COMPLETE')
